### 1. Extracción
Dado que muchos sistemas de tickets (como Zendesk o Freshdesk) exportan archivos `.xls` que en realidad son `XML Spreadsheet 2003`, usar `pd.read_excel()` nos arrojaría un error. Este bloque usa `xml.etree.ElementTree` para parsear ese falso Excel y convertirlo en un DataFrame decente.

In [ ]:
## pip install lxml pandas numpy matplotlib seaborn //si trabajas en VS code instala esto

Note: you may need to restart the kernel to use updated packages.


In [9]:
from lxml import etree
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import re
import unicodedata

# Ruta del archivo
file_path = "9000678639_tickets-April-24-2026-21_21.xls"

print("Iniciando extracción con parser tolerante a fallos")

# El parser con recover=True ignora los errores de sintaxis del XML
parser = etree.XMLParser(recover=True, huge_tree=True)
tree = etree.parse(file_path, parser=parser)
root = tree.getroot()

# El namespace de Microsoft
ns = {'ss': 'urn:schemas-microsoft-com:office:spreadsheet'}

data = []
for row in root.findall('.//ss:Row', namespaces=ns):
    row_data = []
    for cell in row.findall('.//ss:Cell', namespaces=ns):
        data_elem = cell.find('ss:Data', namespaces=ns)
        row_data.append(data_elem.text if data_elem is not None else None)

    # Solo agregamos la fila si tiene al menos un dato (evita filas fantasma)
    if any(row_data):
        data.append(row_data)

print("XML procesado. Construyendo DataFrame...")

# Creamos el DataFrame

if len(data) > 1:
    df_raw = pd.DataFrame(data[1:], columns=data[0])
    print(f"¡Éxito! Dimensiones iniciales: {df_raw.shape}")
    display(df_raw.head(3))
else:
    print("Ocurrió un error, no se encontraron datos.")

Iniciando extracción con parser tolerante a fallos
XML procesado. Construyendo DataFrame...
¡Éxito! Dimensiones iniciales: (10162, 45)


,ID del ticket,Asunto,Estado,Prioridad,Fuente,Tipo,Agente,Grupo,Tiempo de creación,Vencidas hasta ahora,...,Jefe de TI que aprueba la solicitud,Estado de la aprobación,Contacto,Teléfono,Ubicación del servicio,Hora de inicio de la cita,Hora de finalización de la cita,Firma del cliente,Nombre completo,ID de contacto
0,48543,URGENTE- MAS VENTAS MAS PREMIOS,Closed,Low,Portal,Solicitud Operativa,No Agent,No Group,2024-01-02 08:55:58,2024-01-03 18:00:00,...,NaN,NaN,None,None,None,None,None,None,Alexia Trujillo,atrujillom@promotick.com
1,48544,Fuera de oficina Re: ANULAR TRANSACCIÓN BENEFIT,Closed,Low,Email,NaN,No Agent,No Group,2024-01-02 09:27:55,2024-01-04 09:27:55,...,NaN,NaN,None,None,None,None,None,None,Francis Vargas,fvargas@promotick.com
2,48545,enviar pedido no procesado,Closed,Low,Portal,Solicitud Operativa,No Agent,No Group,2024-01-02 09:30:09,2024-01-04 09:30:09,...,NaN,NaN,None,None,None,None,None,None,jassmin Ramirez,jramirez@promotick.com


In [10]:
df_raw.describe()

,ID del ticket,Asunto,Estado,Prioridad,Fuente,Tipo,Agente,Grupo,Tiempo de creación,Vencidas hasta ahora,...,Jefe de TI que aprueba la solicitud,Estado de la aprobación,Contacto,Teléfono,Ubicación del servicio,Hora de inicio de la cita,Hora de finalización de la cita,Firma del cliente,Nombre completo,ID de contacto
count,10162,10161,10162,10162,10162,10094,10162,10162,10162,10162,...,4869,4869,0,0,0,0,0,0,10162,10162
unique,10162,7344,2,4,3,11,9,6,10156,9594,...,3,4,0,0,0,0,0,0,207,228
top,48543,Plantilla Enjoy,Closed,Low,Portal,Solicitud Operativa,No Agent,No Group,2024-05-11 16:23:42,2024-11-20 18:00:00,...,No requiere aprobación,No aplica,NaN,NaN,NaN,NaN,NaN,NaN,Doménica Aguirre,comercial2@motivasoluciones.com
freq,1,211,10155,8177,9465,4397,8309,5286,3,48,...,3320,3320,NaN,NaN,NaN,NaN,NaN,NaN,416,416


In [11]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 10162 entries, 0 to 10161
Data columns (total 45 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   ID del ticket                          10162 non-null  str   
 1   Asunto                                 10161 non-null  str   
 2   Estado                                 10162 non-null  str   
 3   Prioridad                              10162 non-null  str   
 4   Fuente                                 10162 non-null  str   
 5   Tipo                                   10094 non-null  str   
 6   Agente                                 10162 non-null  str   
 7   Grupo                                  10162 non-null  str   
 8   Tiempo de creación                     10162 non-null  str   
 9   Vencidas hasta ahora                   10162 non-null  str   
 10  Tiempo de resolución                   10162 non-null  str   
 11  Hora de cierre            

### Fase 1: Limpieza General y Normalización (Nivel I)

Este bloque de código ejecuta la etapa inicial de preparación y homogeneización del DataFrame, asegurando la consistencia en los metadatos y la estructura de los datos antes de proceder con el análisis.

**Operaciones realizadas:**

* **Preservación de datos:** Crea una copia de seguridad (`df_nivel1`) a partir de `df_raw` para mantener intactos los datos de origen.
* **Normalización de columnas:** Transforma los nombres de las variables al estándar `snake_case` (minúsculas, eliminación de acentos, diéresis y sustitución de caracteres especiales o espacios por guiones bajos).
* **Depuración de cadenas:** Elimina los espacios en blanco iniciales y finales en los campos de tipo texto (`.strip()`).
* **Estandarización de valores nulos:** Reemplaza marcadores de posición textuales (como `"None"`, `"NaN"` o celdas vacías) por valores nulos oficiales de NumPy (`np.nan`).
* **Reducción de dimensionalidad:** Detecta y elimina aquellas columnas que carecen por completo de información (`how='all'`).
* **Validación:** Imprime un resumen de control con las dimensiones resultantes y la estructura técnica actual del DataFrame.

In [12]:
# Hacemos una copia para no arruinar el df_raw por si acaso
df_nivel1 = df_raw.copy()

# 1. Función para transformar los nombres a snake_case y sin tildes
def clean_column_name(name):
    # Quitar acentos y diéresis
    name = ''.join(c for c in unicodedata.normalize('NFD', name) if unicodedata.category(c) != 'Mn')
    # Minúsculas y reemplazar espacios o caracteres extraños por guiones bajos
    name = name.lower().strip()
    name = re.sub(r'[\s\-/\\\(\)\.,\?]+', '_', name)
    name = re.sub(r'_+', '_', name) # Evitar dobles guiones bajos un__nombre
    return name.strip('_')

df_nivel1.columns = [clean_column_name(col) for col in df_nivel1.columns]

# 2. Limpieza de espacios en blanco invisibles dentro de los strings
df_nivel1 = df_nivel1.apply(lambda s: s.str.strip() if s.dtype == "object" else s)

# Reemplazar los strings que simulan estar vacíos o dicen "None"/"NaN" por NaNs reales de Numpy
placeholders_nulos = ["None", "NaN", "nan", "", "null", "Null"]
df_nivel1.replace(placeholders_nulos, np.nan, inplace=True)

# 3. Eliminar columnas que están completamente vacías (axis=1, how='all')
cols_antes = df_nivel1.shape[1]
df_nivel1.dropna(how='all', axis=1, inplace=True)
cols_despues = df_nivel1.shape[1]

print(f"Fase 1 completada.")
print(f"Columnas eliminadas por inservibles (100% nulas): {cols_antes - cols_despues}")
print(f"Nuevas dimensiones del DataFrame: {df_nivel1.shape}")
print("\n--- Columnas resultantes en el DataFrame ---")
print(df_nivel1.columns.tolist())

# Validar el impacto del Nivel I
print("\n--- Inspección Post-Nivel I ---")
df_nivel1.info()

Fase 1 completada.
Columnas eliminadas por inservibles (100% nulas): 8
Nuevas dimensiones del DataFrame: (10162, 37)

--- Columnas resultantes en el DataFrame ---
['id_del_ticket', 'asunto', 'estado', 'prioridad', 'fuente', 'tipo', 'agente', 'grupo', 'tiempo_de_creacion', 'vencidas_hasta_ahora', 'tiempo_de_resolucion', 'hora_de_cierre', 'ultima_hora_de_la_actualizacion', 'tiempo_de_respuesta_inicial', 'tiempo_de_seguimiento', 'primer_tiempo_de_respuesta_en_horas', 'tiempo_de_resolucion_en_horas', 'interacciones_de_agente', 'interacciones_de_cliente', 'estado_de_resolucion', 'estado_de_la_primera_respuesta', 'etiquetas', 'resultados_de_la_encuesta', 'producto', 'tiempo_empleado', 'inicio_de_atencion', 'fin_de_atencion', 'tipo_de_solicitud', 'empresa', 'tipo_de_programa', 'origen', 'origen_categoria', 'dependencias', 'jefe_de_ti_que_aprueba_la_solicitud', 'estado_de_la_aprobacion', 'nombre_completo', 'id_de_contacto']

--- Inspección Post-Nivel I ---
<class 'pandas.DataFrame'>
RangeIndex

### Eliminar variables descontinuadas

Eliminar variables que no aportan valor analítico o que han sido descontinuadas según lo indicado por el representante de Promotick. Esto permite reducir ruido en los datos y mantener únicamente información relevante para el análisis.

Se eliminan las siguientes variables:

- "tiempo_de_seguimiento": no utilizada en los procesos actuales.
- "estado_de_la_aprobación": campo asociado a aprobaciones no relevante  para el análisis.
- "jefe_de_ti_que_aprueba_la_solicitud": variable administrativa sin valor analítico.
- "producto": columna sin información (vacía).
- "id_de_contacto": identificador no necesario para el análisis de negocio.

In [13]:
df_nivel1.drop(columns=[ "tiempo_de_seguimiento", "estado_de_la_aprobacion", "jefe_de_ti_que_aprueba_la_solicitud","producto","id_de_contacto"],
        inplace=True)

In [14]:
df_nivel1.info()

<class 'pandas.DataFrame'>
RangeIndex: 10162 entries, 0 to 10161
Data columns (total 32 columns):
 #   Column                               Non-Null Count  Dtype
---  ------                               --------------  -----
 0   id_del_ticket                        10162 non-null  str  
 1   asunto                               10161 non-null  str  
 2   estado                               10162 non-null  str  
 3   prioridad                            10162 non-null  str  
 4   fuente                               10162 non-null  str  
 5   tipo                                 10094 non-null  str  
 6   agente                               10162 non-null  str  
 7   grupo                                10162 non-null  str  
 8   tiempo_de_creacion                   10162 non-null  str  
 9   vencidas_hasta_ahora                 10162 non-null  str  
 10  tiempo_de_resolucion                 10162 non-null  str  
 11  hora_de_cierre                       10155 non-null  str  
 12  u

### Fase 2: Casteo de Tipos y Tratamiento de Duraciones (Nivel II)

Este bloque de código ejecuta la segunda etapa de preparación de los datos, enfocada en la conversión estricta de tipos de datos (*type casting*) y en la transformación de cadenas de texto complejas en métricas numéricas analizables.

**Operaciones realizadas:**

* **Preservación de datos:** Modifica una nueva copia de seguridad (`df_nivel2`) a partir del DataFrame normalizado en la fase anterior.
* **Conversión cronológica:** Transforma las columnas analógicas de marcas de tiempo (`cols_fechas`) al formato nativo `datetime64[ns]` de Pandas, forzando como nulos (`errors='coerce'`) los registros con errores de sintaxis.
* **Preservación de identificadores:** Convierte la variable `id_del_ticket` a tipo cadena de texto (`str`) para evitar cálculos aritméticos accidentales o pérdida de ceros a la izquierda.
* **Homogeneización de contadores:** Fuerza las variables de conteo de interacciones a tipo numérico y las almacena bajo el formato `Int64` (entero nativo de Pandas que admite valores nulos).
* **Procesamiento de duraciones textuales:** Identifica las columnas con formato de tiempo compuesto (`0 days HH:MM:SS`), las convierte temporalmente a `timedelta` y calcula el equivalente total en horas con precisión decimal (`float`).
* **Validación de métricas:** Asegura que los indicadores numéricos continuos (como `tiempo_empleado`) queden tipificados correctamente como punto flotante (`float64`).
* **Auditoría:** Muestra una previsualización de los cálculos de tiempo y despliega la nueva estructura técnica consolidada del dataset.

In [15]:
df_nivel2 = df_nivel1.copy()

print("Ejecutando Nivel II: Tratamiento de duraciones encubiertas...")

# 1. Fechas puras (Datetime)
cols_fechas = [
    'tiempo_de_creacion',
    'vencidas_hasta_ahora',
    'hora_de_cierre',
    'ultima_hora_de_la_actualizacion',
    'inicio_de_atencion',
    'fin_de_atencion'
]
for col in cols_fechas:
    df_nivel2[col] = pd.to_datetime(df_nivel2[col], errors='coerce')

# 2. Identificadores categóricos / Texto seguro
df_nivel2['id_del_ticket'] = df_nivel2['id_del_ticket'].astype(str)


# 3. Conteos de interacciones (Enteros con soporte de NaNs)
cols_enteros = ['interacciones_de_agente', 'interacciones_de_cliente']
for col in cols_enteros:
    df_nivel2[col] = pd.to_numeric(df_nivel2[col], errors='coerce').astype('Int64')

# 4. Duraciones que dicen "en horas" pero vienen como '0 days HH:MM:SS'
cols_duracion_texto = ['primer_tiempo_de_respuesta_en_horas', 'tiempo_de_resolucion_en_horas']
for col in cols_duracion_texto:
    # Convertimos a timedelta y luego extraemos el total de horas reales en formato float
    timedelta_temp = pd.to_timedelta(df_nivel2[col], errors='coerce')
    df_nivel2[col] = timedelta_temp.dt.total_seconds() / 3600.0

# 5. Métricas numéricas que ya son floats puros
df_nivel2['tiempo_empleado'] = pd.to_numeric(df_nivel2['tiempo_empleado'], errors='coerce')

print("¡Casteo completado!")
print(f"Dimensiones del DataFrame: {df_nivel2.shape}")

# Verificar que las horas flotantes ahora sí tengan sentido y no sean nulos
print("\n--- Muestra de las horas calculadas (primeras 3 filas) ---")
print(df_nivel2[cols_duracion_texto].head(3))

print("\n--- Estructura final del dataset para Nivel III ---")
df_nivel2.info()

Ejecutando Nivel II: Tratamiento de duraciones encubiertas...
¡Casteo completado!
Dimensiones del DataFrame: (10162, 32)

--- Muestra de las horas calculadas (primeras 3 filas) ---
   primer_tiempo_de_respuesta_en_horas  tiempo_de_resolucion_en_horas
0                             0.256389                       0.260000
1                             0.000000                       0.033333
2                             0.617222                      20.044167

--- Estructura final del dataset para Nivel III ---
<class 'pandas.DataFrame'>
RangeIndex: 10162 entries, 0 to 10161
Data columns (total 32 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   id_del_ticket                        10162 non-null  str           
 1   asunto                               10161 non-null  str           
 2   estado                               10162 non-null  str           
 3   prioridad         

### Fase 3: Auditoría de Calidad y Consistencia de Negocio (Nivel III)

Este bloque de código ejecuta una fase de diagnóstico avanzado sobre la integridad de los datos. Se enfoca en cuantificar el impacto de los valores faltantes y en validar mediante lógica de negocio si estas ausencias responden a un comportamiento esperado o a anomalías en el registro.

**Operaciones realizadas:**

* **Preservación de datos:** Genera una nueva copia de seguridad (`df_nivel3`) para aislar las pruebas de diagnóstico de las fases de transformación previas.
* **Cuantificación de valores faltantes:** Calcula la frecuencia absoluta y el impacto porcentual de valores nulos (`NaN`) por cada variable, ordenando los resultados de manera descendente y filtrando únicamente aquellas columnas que presentan deficiencias.
* **Validación de reglas de negocio (Caso 1 - Ciclo de vida):** Analiza la distribución de la variable `estado` en aquellos registros donde `hora_de_cierre` es nula, permitiendo verificar si la ausencia de la fecha coincide correctamente con tickets que aún permanecen abiertos o en proceso.
* **Validación de reglas de negocio (Caso 2 - Correlación de nulos):** Evalúa la consistencia simultánea de las variables `origen` y `origen_categoria` para determinar si el vacío de información ocurre de forma indexada en las mismas filas, ayudando a identificar dependencias estructurales o fallos en el flujo de captura del sistema de TI.

In [16]:
df_nivel3 = df_nivel2.copy()

# 1. Calcular frecuencias y porcentajes de valores faltantes
missing_summary = pd.DataFrame({
    'Total Nulos': df_nivel3.isna().sum(),
    'Porcentaje (%)': (df_nivel3.isna().sum() / len(df_nivel3)) * 100
}).sort_values(by='Total Nulos', ascending=False)

# Nos enfocamos únicamente en las columnas que sufren de vacíos
missing_summary = missing_summary[missing_summary['Total Nulos'] > 0]

print("--- REPORTE DE CALIDAD: COLUMNAS CON VALORES FALTANTES ---")
display(missing_summary)

# 2. Prueba de Coherencia de Negocio N°1: ¿Por qué falta 'hora_de_cierre'?
# Sospecha: Si el ticket no está cerrado ('Closed' / 'Resolved'), no debería tener hora de cierre.
print("\n--- PRUEBA DE COHERENCIA: 'hora_de_cierre' vs 'estado' ---")
print(df_nivel3[df_nivel3['hora_de_cierre'].isna()]['estado'].value_counts())

# 3. Prueba de Coherencia de Negocio N°2: El misterio del bloque de TI
# Hay 2 columnas (origen, origen_categoria)
# que parecen tener exactamente la misma cantidad de nulos. Veamos si fallan en las mismas filas.
cols_ti = ['origen', 'origen_categoria']
print("\n--- PRUEBA DE COHERENCIA: Bloque de Aprobaciones de TI ---")
# Contamos cuántas filas tienen nulas ESTAS DOS columnas al mismo tiempo
nulos_simultaneos = df_nivel3[cols_ti].isna().all(axis=1).sum()
print(f"Filas donde las 2 columnas de aprobación son nulas simultáneamente: {nulos_simultaneos} de {len(df_nivel3)}")

--- REPORTE DE CALIDAD: COLUMNAS CON VALORES FALTANTES ---


,Total Nulos,Porcentaje (%)
etiquetas,9782,96.260579
resultados_de_la_encuesta,9772,96.162173
dependencias,8252,81.204487
origen,5293,52.086204
origen_categoria,5293,52.086204
tipo_de_solicitud,2684,26.412124
tiempo_de_respuesta_inicial,703,6.917930
estado_de_la_primera_respuesta,703,6.917930
tipo_de_programa,376,3.700059
fin_de_atencion,117,1.151348



--- PRUEBA DE COHERENCIA: 'hora_de_cierre' vs 'estado' ---
estado
Resolved    7
Name: count, dtype: int64

--- PRUEBA DE COHERENCIA: Bloque de Aprobaciones de TI ---
Filas donde las 2 columnas de aprobación son nulas simultáneamente: 5293 de 10162


### Fase 3: Imputación Estratégica y Resolución de Nulos (Nivel III)

Este bloque de código ejecuta la estrategia de limpieza avanzada e imputación de datos faltantes, aplicando criterios de lógica de negocio, rellenado condicional y reparación temporal para eliminar la presencia de valores nulos (`NaN`) en el dataset.

**Operaciones realizadas:**

* **Imputación de variables estructurales (MAR):** Identifica las columnas del bloque de TI (`origen`, `origen_categoria`) que presentan vacíos estructurales y las estandariza bajo la etiqueta `'No aplica'`, reconociendo que la falta de datos responde a la naturaleza del flujo y no a un error de registro.
* **Homogeneización de categorías dispersas o condicionales:** Sustituye los valores faltantes en variables cualitativas complementarias (como `etiquetas`, `resultados_de_la_encuesta`, `dependencias`, entre otras) por valores por defecto (`'Sin etiqueta'`, `'Ninguna'`, `'No especificado'`), preservando la representatividad de la muestra sin perder registros en el filtrado.
* **Reparación de consistencia temporal:** Corrige inconsistencias cronológicas en el ciclo de vida del ticket mediante imputación cruzada; asigna la última fecha de actualización a los cierres ausentes, y homologa las fechas de inicio y fin de atención con los tiempos de creación y cierre si estos no fueron registrados.
* **Imputación numérica condicional:** Reemplaza los vacíos en la métrica continua `tiempo_empleado` utilizando un enfoque segmentado: se calcula e imputa la mediana móvil de la variable agrupada específicamente por el nivel de `prioridad` del ticket, evitando el sesgo de una media global.
* **Control de calidad final:** Evalúa la efectividad de la purga imprimiendo el conteo total de valores nulos remanentes y validando si el dataset ha alcanzado con éxito el 0% de registros nulos.

In [17]:
df_nivel3 = df_nivel2.copy()

print("Iniciando Imputación Estratégica de Nivel III...")

# 1. Tratamiento del Bloque de TI (MAR Estructural)
cols_ti = ['origen', 'origen_categoria']
for col in cols_ti:
    df_nivel3[col] = df_nivel3[col].fillna('No aplica')

# 2. Tratamiento de variables categóricas opcionales o altamente dispersas
df_nivel3['etiquetas'] = df_nivel3['etiquetas'].fillna('Sin etiqueta')
df_nivel3['resultados_de_la_encuesta'] = df_nivel3['resultados_de_la_encuesta'].fillna('Sin encuesta')
df_nivel3['dependencias'] = df_nivel3['dependencias'].fillna('Ninguna')
df_nivel3['tipo_de_solicitud'] = df_nivel3['tipo_de_solicitud'].fillna('No especificado')
df_nivel3['tiempo_de_respuesta_inicial'] = df_nivel3['tiempo_de_respuesta_inicial'].fillna('Sin respuesta')
df_nivel3['estado_de_la_primera_respuesta'] = df_nivel3['estado_de_la_primera_respuesta'].fillna('Sin respuesta')

# Limpieza de nulos menores en otras categorías
cols_cat_menores = ['tipo_de_programa', 'empresa', 'tipo', 'asunto']
for col in cols_cat_menores:
    df_nivel3[col] = df_nivel3[col].fillna('No especificado')

# 3. Reparación de Coherencia Temporal
# Si está resuelto pero no hay hora de cierre, asumimos la última actualización como hora de cierre
df_nivel3['hora_de_cierre'] = df_nivel3['hora_de_cierre'].fillna(df_nivel3['ultima_hora_de_la_actualizacion'])

# Lo mismo para inicio y fin de atención: si faltan, usamos creación y cierre para mantener coherencia
df_nivel3['inicio_de_atencion'] = df_nivel3['inicio_de_atencion'].fillna(df_nivel3['tiempo_de_creacion'])
df_nivel3['fin_de_atencion'] = df_nivel3['fin_de_atencion'].fillna(df_nivel3['hora_de_cierre'])

# 4. Imputación Condicional Avanzada (Métricas Numéricas)
# En lugar de la media global, imputamos la mediana de 'tiempo_empleado' según la 'prioridad' del ticket
mediana_por_prioridad = df_nivel3.groupby('prioridad')['tiempo_empleado'].transform('median')
df_nivel3['tiempo_empleado'] = df_nivel3['tiempo_empleado'].fillna(mediana_por_prioridad)

print("¡Estrategia de imputación completada!")
print(f"Total de valores nulos remanentes en todo el dataset: {df_nivel3.isna().sum().sum()}")

# Validar si queda alguna columna con nulos
nulos_por_columna = df_nivel3.isna().sum()
if nulos_por_columna.sum() > 0:
    print("\n--- Columnas que se resisten a la purga ---")
    print(nulos_por_columna[nulos_por_columna > 0])
else:
    print("\nDataset con 0% nulos.")

Iniciando Imputación Estratégica de Nivel III...
¡Estrategia de imputación completada!
Total de valores nulos remanentes en todo el dataset: 0

Dataset con 0% nulos.


In [18]:
df_nivel3.info()

<class 'pandas.DataFrame'>
RangeIndex: 10162 entries, 0 to 10161
Data columns (total 32 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   id_del_ticket                        10162 non-null  str           
 1   asunto                               10162 non-null  str           
 2   estado                               10162 non-null  str           
 3   prioridad                            10162 non-null  str           
 4   fuente                               10162 non-null  str           
 5   tipo                                 10162 non-null  str           
 6   agente                               10162 non-null  str           
 7   grupo                                10162 non-null  str           
 8   tiempo_de_creacion                   10162 non-null  datetime64[us]
 9   vencidas_hasta_ahora                 10162 non-null  datetime64[us]
 10  tiempo_de_resolucion 

In [19]:
df_nivel3['tiempo_de_respuesta_inicial']

0        2024-01-02 09:15:23
1              Sin respuesta
2        2024-01-02 10:07:11
3              Sin respuesta
4        2024-01-02 13:35:22
                ...         
10157    2026-01-02 13:54:16
10158    2026-01-02 17:46:49
10159    2026-01-02 17:48:10
10160    2026-01-06 15:21:45
10161    2025-12-31 14:29:10
Name: tiempo_de_respuesta_inicial, Length: 10162, dtype: str

### Casteo Estricto y Optimización de Memoria

Este bloque de código consolida el dataset en su estado definitivo mediante un tipado estricto (*strict type casting*). El objetivo es asegurar la máxima eficiencia en el uso de memoria RAM y garantizar la compatibilidad de los tipos de datos con algoritmos de modelamiento o herramientas de visualización.

**Operaciones realizadas:**

* **Tratamiento cronológico avanzado y resolución de excepciones:** Convierte las marcas de tiempo restantes al formato nativo `datetime64[ns]`. Para la variable `tiempo_de_respuesta_inicial` (que contenía cadenas textuales como `'Sin respuesta'`), fuerza la conversión enviando los errores a `NaT` (*Not a Time*) y repara inmediatamente dichos registros imputándoles la fecha de `tiempo_de_creacion`.
* **Indexación de identificadores:** Asegura la persistencia de la variable `id_del_ticket` de forma explícita como tipo cadena de texto (`str`).
* **Optimización dimensional (Casteo Categórico):** Transforma un bloque masivo de 18 variables cualitativas (como `estado`, `prioridad`, `agente`, entre otras) de tipo genérico `object` al tipo de datos optimizado `category`. Esto reduce drásticamente el consumo de memoria del DataFrame al almacenar las cadenas repetitivas como diccionarios de punteros enteros.
* **Consolidación de métricas cuantitativas:** Refuerza el tipado de los indicadores de tiempo y rendimiento como números de punto flotante de alta precisión (`float64`).
* **Preservación de consistencia en enteros:** Consolida las variables de conteo de interacciones bajo el formato `Int64`, manteniendo la compatibilidad estructural con posibles ausencias de datos a nivel macro.
* **Auditoría final:** Despliega un reporte completo mediante `.info()` para validar la estructura técnica definitiva, los tipos asignados y las dimensiones del dataset listo para producción.

In [20]:
print("Iniciando Fase Final: Casteo Estricto de las 32 variables actuales...")

df = df_nivel3.copy()

# =========================================================================
# 2. PARSEO DE FECHAS Y TRATAMIENTO DE "SIN RESPUESTA"
# =========================================================================

# Primero parseamos la que variable que no da problemas ('tiempo_de_resolucion')
df['tiempo_de_resolucion'] = pd.to_datetime(df['tiempo_de_resolucion'], errors='coerce')

# Convertimos la variable problemática forzando el error (los "Sin respuesta" se vuelven NaT temporalmente)
df['tiempo_de_respuesta_inicial'] = pd.to_datetime(df['tiempo_de_respuesta_inicial'], errors='coerce')

# Rellenamos esos NaT usando el valor de 'tiempo_de_creacion' para mantener el 100% de los datos
df['tiempo_de_respuesta_inicial'] = df['tiempo_de_respuesta_inicial'].fillna(df['tiempo_de_creacion'])


# =========================================================================
# 3. CASTEO ESTRICTO A CATEGORÍAS Y TIPOS DEFINITIVOS
# =========================================================================

# Identificador único como string
df['id_del_ticket'] = df['id_del_ticket'].astype(str)

# Variables Categóricas (Las 18 restantes que son texto)
cols_categoricas = [
    'asunto', 'estado', 'prioridad', 'fuente', 'tipo', 'agente', 'grupo',
    'estado_de_resolucion', 'estado_de_la_primera_respuesta', 'etiquetas',
    'resultados_de_la_encuesta', 'tipo_de_solicitud', 'empresa', 'tipo_de_programa',
    'origen', 'origen_categoria', 'dependencias', 'nombre_completo'
]

for col in cols_categoricas:
    df[col] = df[col].astype('category')

# Numéricas Continuas (Floats)
cols_floats = ['primer_tiempo_de_respuesta_en_horas', 'tiempo_de_resolucion_en_horas', 'tiempo_empleado']
for col in cols_floats:
    df[col] = df[col].astype('float64')

# Numéricas Enteras (Manteniendo Int64 por consistencia macro)
cols_enteros = ['interacciones_de_agente', 'interacciones_de_cliente']
for col in cols_enteros:
    df[col] = df[col].astype('Int64')

print("\n¡Casteo completado con éxito!")
print(f"Dimensiones finales del DataFrame: {df.shape}")
print("\n--- Inspección Técnica Estricta Post-Casteo ---")
df.info()

Iniciando Fase Final: Casteo Estricto de las 32 variables actuales...

¡Casteo completado con éxito!
Dimensiones finales del DataFrame: (10162, 32)

--- Inspección Técnica Estricta Post-Casteo ---
<class 'pandas.DataFrame'>
RangeIndex: 10162 entries, 0 to 10161
Data columns (total 32 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   id_del_ticket                        10162 non-null  str           
 1   asunto                               10162 non-null  category      
 2   estado                               10162 non-null  category      
 3   prioridad                            10162 non-null  category      
 4   fuente                               10162 non-null  category      
 5   tipo                                 10162 non-null  category      
 6   agente                               10162 non-null  category      
 7   grupo                                10162 

### Fase de Control de Calidad: Duplicados y Coherencia de Negocio

Este bloque de código ejecuta una auditoría de integridad avanzada sobre el dataset definitivo. Su propósito es detectar redundancias estructurales, validar la consistencia lógica entre variables categóricas correlacionadas y asegurar la coherencia cronológica de los flujos de tiempo.

**Operaciones realizadas:**

* **Detección de redundancias y colisiones de ID:** Cuantifica la existencia de registros idénticos en su totalidad (`.duplicated()`) y evalúa si existen identificadores únicos (`id_del_ticket`) duplicados con atributos diferentes, lo cual alertaría sobre inconsistencias en la extracción o almacenamiento de la base de datos.
* **Análisis de concordancia mediante Tablas de Contingencia:** Construye una matriz de cruce (`pd.crosstab`) entre las variables categóricas `estado` y `estado_de_resolucion`. Esto permite auditar visualmente si existen contradicciones lógicas (por ejemplo, tickets marcados operativamente como `'Abierto'` pero con un estado de resolución de `'Resuelto'`).
* **Validación de reglas cronológicas:** Implementa una prueba de consistencia temporal que identifica si existen anomalías lógicas en el ciclo de vida de los registros, específicamente verificando si alguna fecha de `hora_de_cierre` se registró de manera errónea antes de su respectivo `tiempo_de_creacion`.

In [21]:
print("--- CONTROL DE CALIDAD: DUPLICADOS Y COHERENCIA ---")

# 1. Buscar clones exactos y IDs repetidos
duplicados_totales = df.duplicated().sum()
duplicados_id = df.duplicated(subset=['id_del_ticket']).sum()

print(f"Filas clonadas 100% idénticas: {duplicados_totales}")
print(f"Tickets con el mismo ID pero datos distintos: {duplicados_id}")

# 2. Matriz de Coherencia Categórica
# Vamos a cruzar el 'estado' del ticket con el 'estado_de_resolucion'
print("\n--- MATRIZ DE CRUCE: 'estado' vs 'estado_de_resolucion' ---")
matriz_coherencia = pd.crosstab(
    df['estado'],
    df['estado_de_resolucion'],
    margins=True
)
display(matriz_coherencia)

# 3. Verificación de Fechas Invertidas
# Un ticket no puede cerrarse antes de crearse. Busquemos si hay errores de lógica temporal.
viajeros_del_tiempo = df[df['hora_de_cierre'] < df['tiempo_de_creacion']].shape[0]
print(f"\nTickets donde la fecha de cierre es ANTERIOR a la de creación: {viajeros_del_tiempo}")

--- CONTROL DE CALIDAD: DUPLICADOS Y COHERENCIA ---
Filas clonadas 100% idénticas: 0
Tickets con el mismo ID pero datos distintos: 0

--- MATRIZ DE CRUCE: 'estado' vs 'estado_de_resolucion' ---


estado_de_resolucion,SLA Violated,Within SLA,All
estado,,,
Closed,3152,7003,10155
Resolved,3,4,7
All,3155,7007,10162



Tickets donde la fecha de cierre es ANTERIOR a la de creación: 0


# AQUI TERMINA PARTE JUAN VELO

OUTLIERS

Integración, reducción y transformación de datos

Feature Engineering: variables derivadas y atributos construidos

Análisis y visualización de datos para búsqueda de insights